# 01 - Data Collection
This notebook fetches data from the official Fantasy Premier League API.
We collect player statistics, team information, and gameweek-by-gameweek history.

In [1]:
import os
import json
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

BASE_URL = 'https://fantasy.premierleague.com/api'
RAW_DIR = '../data/raw'
PROCESSED_DIR = '../data/processed'

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print('Directories ready.')

Directories ready.


In [2]:
# Fetch bootstrap data (all players, teams, gameweeks)
response = requests.get(f'{BASE_URL}/bootstrap-static/', timeout=30)
bootstrap = response.json()

with open(f'{RAW_DIR}/bootstrap.json', 'w') as f:
    json.dump(bootstrap, f)

print(f'Players found: {len(bootstrap["elements"])}')
print(f'Teams found: {len(bootstrap["teams"])}')
print(f'Gameweeks found: {len(bootstrap["events"])}')

Players found: 564
Teams found: 20
Gameweeks found: 38


In [3]:
# Build player summary DataFrame
position_map = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
teams = {t['id']: t['name'] for t in bootstrap['teams']}

players = []
for p in bootstrap['elements']:
    players.append({
        'id': p['id'],
        'name': p['web_name'],
        'team': teams.get(p['team'], 'Unknown'),
        'position': position_map.get(p['element_type'], 'Unknown'),
        'price': p['now_cost'] / 10,
        'total_points': p['total_points'],
        'minutes': p['minutes'],
        'goals_scored': p['goals_scored'],
        'assists': p['assists'],
        'clean_sheets': p['clean_sheets'],
        'goals_conceded': p['goals_conceded'],
        'yellow_cards': p['yellow_cards'],
        'red_cards': p['red_cards'],
        'saves': p['saves'],
        'bonus': p['bonus'],
        'bps': p['bps'],
        'influence': float(p['influence']),
        'creativity': float(p['creativity']),
        'threat': float(p['threat']),
        'ict_index': float(p['ict_index']),
        'form': float(p['form']),
        'points_per_game': float(p['points_per_game']),
        'selected_by_percent': float(p['selected_by_percent']),
        'transfers_in': p['transfers_in'],
        'transfers_out': p['transfers_out'],
    })

players_df = pd.DataFrame(players)
players_df.to_csv(f'{PROCESSED_DIR}/players_season.csv', index=False)
print(players_df.shape)
players_df.head()

(564, 25)


,id,name,team,position,price,total_points,minutes,goals_scored,assists,clean_sheets,...,bps,influence,creativity,threat,ict_index,form,points_per_game,selected_by_percent,transfers_in,transfers_out
0,1,Raya,Arsenal,GKP,6.0,162,3330,0,0,19,...,633,541.6,33.5,0.0,57.5,0.0,4.4,30.4,0,0
1,2,Arrizabalaga,Arsenal,GKP,5.0,2,90,0,0,0,...,11,19.8,0.0,0.0,2.0,0.0,2.0,0.2,0,0
2,3,Meslier,Arsenal,GKP,5.0,0,0,11,0,0,...,11,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0,0
3,4,Gabriel,Arsenal,DEF,8.0,209,2750,3,5,18,...,724,824.0,128.5,298.0,125.0,0.0,6.5,25.5,0,0
4,5,J.Timber,Arsenal,DEF,6.5,149,2452,3,6,13,...,532,499.0,402.6,352.0,125.7,0.0,5.0,0.6,0,0


In [4]:
# Fetch gameweek history for every player
all_gw = []
player_ids = players_df['id'].tolist()

for pid in tqdm(player_ids, desc='Fetching histories'):
    try:
        r = requests.get(f'{BASE_URL}/element-summary/{pid}/', timeout=30)
        history = r.json().get('history', [])
        for gw in history:
            gw['player_id'] = pid
            all_gw.append(gw)
        time.sleep(0.3)
    except Exception as e:
        print(f'Failed for player {pid}: {e}')

gw_df = pd.DataFrame(all_gw)
gw_df = gw_df.merge(players_df[['id','name','team','position','price']],
                     left_on='player_id', right_on='id', how='left')

gw_df.to_csv(f'{PROCESSED_DIR}/player_gameweek_history.csv', index=False)
print(f'Total gameweek records: {len(gw_df)}')
gw_df.head()

Fetching histories:   0%|          | 0/564 [00:00<?, ?it/s]

Failed for player 149: HTTPSConnectionPool(host='fantasy.premierleague.com', port=443): Max retries exceeded with url: /api/element-summary/149/ (Caused by ConnectTimeoutError(<HTTPSConnection(host='fantasy.premierleague.com', port=443) at 0x2a79f2d5c90>, 'Connection to fantasy.premierleague.com timed out. (connect timeout=30)'))


KeyError: 'player_id'

In [5]:
# Debug - check how many records fetched
print(f"Records in all_gw: {len(all_gw)}")

# Test single player fetch
r = requests.get(f'{BASE_URL}/element-summary/1/', timeout=30)
print(f"Status code: {r.status_code}")
print(r.json().keys())

Records in all_gw: 0
Status code: 200
dict_keys(['fixtures', 'history', 'history_past'])


In [6]:
# Test one player history directly
r = requests.get(f'{BASE_URL}/element-summary/1/', timeout=30)
history = r.json().get('history', [])
print(f"History records for player 1: {len(history)}")
if history:
    print(history[0])

History records for player 1: 0


In [7]:
# Check past season data for player 1
r = requests.get(f'{BASE_URL}/element-summary/1/', timeout=30)
data = r.json()
print(f"History (current season): {len(data['history'])}")
print(f"History past (previous seasons): {len(data['history_past'])}")
if data['history_past']:
    print(data['history_past'][0])

History (current season): 0
History past (previous seasons): 5
{'season_name': '2021/22', 'element_code': 154561, 'start_cost': 45, 'end_cost': 44, 'total_points': 95, 'minutes': 2160, 'goals_scored': 0, 'assists': 0, 'clean_sheets': 8, 'goals_conceded': 27, 'own_goals': 0, 'penalties_saved': 0, 'penalties_missed': 0, 'yellow_cards': 1, 'red_cards': 0, 'saves': 78, 'bonus': 5, 'bps': 496, 'influence': '593.4', 'creativity': '10.0', 'threat': '0.0', 'ict_index': '60.1', 'clearances_blocks_interceptions': 0, 'recoveries': 0, 'tackles': 0, 'defensive_contribution': 0, 'starts': 0, 'expected_goals': '0.00', 'expected_assists': '0.00', 'expected_goal_involvements': '0.00', 'expected_goals_conceded': '0.00'}


In [8]:
# Fetch past season data for all players
all_past = []
player_ids = players_df['id'].tolist()

for pid in tqdm(player_ids, desc='Fetching past seasons'):
    try:
        r = requests.get(f'{BASE_URL}/element-summary/{pid}/', timeout=30)
        past = r.json().get('history_past', [])
        for season in past:
            season['player_id'] = pid
            all_past.append(season)
        time.sleep(0.3)
    except Exception as e:
        print(f'Failed for player {pid}: {e}')

past_df = pd.DataFrame(all_past)

# Merge player metadata
past_df = past_df.merge(
    players_df[['id','name','team','position','price']],
    left_on='player_id', right_on='id', how='left'
)

past_df.to_csv(f'{PROCESSED_DIR}/player_past_seasons.csv', index=False)
print(f'Total records: {len(past_df)}')
past_df.head()

Fetching past seasons:   0%|          | 0/564 [00:00<?, ?it/s]

Total records: 2017


,season_name,element_code,start_cost,end_cost,total_points,minutes,goals_scored,assists,clean_sheets,goals_conceded,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,player_id,id,name,team,position,price
0,2021/22,154561,45,44,95,2160,0,0,8,27,...,0.00,0.00,0.00,0.00,1,1,Raya,Arsenal,GKP,6.0
1,2022/23,154561,45,48,166,3420,0,0,12,46,...,0.11,0.12,0.23,50.12,1,1,Raya,Arsenal,GKP,6.0
2,2023/24,154561,50,53,135,2880,0,0,16,24,...,0.00,0.04,0.04,22.51,1,1,Raya,Arsenal,GKP,6.0
3,2024/25,154561,55,56,142,3420,0,0,13,34,...,0.00,0.03,0.03,35.03,1,1,Raya,Arsenal,GKP,6.0
4,2025/26,154561,55,62,162,3330,0,0,19,26,...,0.00,0.07,0.07,27.56,1,1,Raya,Arsenal,GKP,6.0
